# LaGCORL pretrained-agent evaluation

Choose `BENCHMARK` once. The notebook builds a fresh randomized evaluation corpus, discovers compatible checkpoints under `logs/`, and evaluates whichever agents are available for that benchmark.

In [ ]:
# Experiment controls — choose the benchmark here, once.
BENCHMARK = "maxcut"  # "maxcut" or "vmp"

## 0. Imports and project paths

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, random, importlib, inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

CWD = Path.cwd().resolve()
if (CWD / "LaGCORL" / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = CWD / "LaGCORL"
elif (CWD / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = CWD
else:
    raise FileNotFoundError(f"Cannot find gae/train_gae.py from {CWD}.")

for entry in (PROJECT_ROOT.parent, PROJECT_ROOT, CWD):
    entry = str(entry.resolve())
    if entry not in sys.path:
        sys.path.insert(0, entry)

NOTEBOOK_DIR = CWD
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Reload the helper module so notebook edits are reflected before functions are imported.
importlib.invalidate_caches()
import notebook_helpers_api as helpers_api
helpers_api = importlib.reload(helpers_api)

if "vmp_range" not in inspect.signature(helpers_api.sample_benchmark_corpus).parameters:
    raise TypeError(
        "The loaded notebook_helpers_api.sample_benchmark_corpus does not accept "
        "'vmp_range'. Check the printed module path and update/reload that file."
    )

from notebook_helpers_api import (
    sample_benchmark_corpus,
    load_benchmark_env,
    newest_benchmark_artifact,
    newest_gae_bundle,
    benchmark_on_corpus,
    build_padded_gnn_discrete_corpus,
    evaluate_padded_gnn_policy,
    create_run_layout,
)
from LaGCORL.agents.notebook_train_gae_api import (
    load_gae_encoders,
    make_agent_training_wrapper_factory,
)
from LaGCORL.agents.notebook_test_api import test_agent_on_env
from LaGCORL.agents.notebook_train_api import _get_model_class

SEED = 7
EVAL_EPISODES = 10
EVAL_SEEDS = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110]
LOG_ROOT = PROJECT_ROOT.parent / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

## 1. Select the benchmark and generate a randomized evaluation suite

In [ ]:
# Select once; the evaluation code below is benchmark-independent.
TEST_SCENARIO_COUNT = 5

# MaxCut Params
TEST_NUM_NODES = 20  
TEST_EDGE_WEIGHT_RANGE = (1, 20)

# VMP Params
TEST_VMP_RANGE = {
    "min_VMs": 4,
    "max_VMs": 10,
    "min_PMs": 7,
    "max_PMs": 30,
}

TEST_CORPUS_SEED = 2026
TEST_MAX_SWEEPS = 100

test_corpus = sample_benchmark_corpus(
    PROJECT_ROOT,
    BENCHMARK,
    count=TEST_SCENARIO_COUNT,
    seed=TEST_CORPUS_SEED,
    max_sweeps=TEST_MAX_SWEEPS,
    graph_size_range=(TEST_NUM_NODES, TEST_NUM_NODES),
    edge_weight_range=TEST_EDGE_WEIGHT_RANGE,
    vmp_range=TEST_VMP_RANGE,
)
TEST_CORPUS_DIR = Path(test_corpus["corpus_dir"])
TEST_ENV_IDS = [int(value) for value in test_corpus["graphs"]["id"].tolist()]

print("Benchmark:", BENCHMARK)
print("Test corpus:", TEST_CORPUS_DIR)
print("Scenario count:", len(TEST_ENV_IDS))
display(test_corpus["graphs"].head(10))


## 2. Discover pretrained artifacts

In [ ]:
ARTIFACTS = {
    "discrete": newest_benchmark_artifact(LOG_ROOT, "discrete_ppo", BENCHMARK),
    "gnn_discrete": newest_benchmark_artifact(LOG_ROOT, "discrete_ppo_gnn_padded", BENCHMARK),
    "iterative": newest_benchmark_artifact(LOG_ROOT, "iterative_dqn", BENCHMARK),
    "projection": newest_benchmark_artifact(LOG_ROOT, "projection_ppo", BENCHMARK),
}
GAE_RUN_DIR = newest_gae_bundle(LOG_ROOT, BENCHMARK)

for name, path in ARTIFACTS.items():
    print(f"{name:>12}: {path if path is not None else 'not found'}")
print("         GAE:", GAE_RUN_DIR if GAE_RUN_DIR is not None else "not found")

if not any(path is not None for path in ARTIFACTS.values()):
    raise FileNotFoundError(
        f"No pretrained policies were found for benchmark={BENCHMARK!r} under {LOG_ROOT}. "
        "Run the setup notebook or train the selected benchmark first."
    )


## 3. Configure GAE-backed wrappers when the encoder bundle is available

In [ ]:
wrapper_factories = {}
if GAE_RUN_DIR is not None:
    gae_bundle = load_gae_encoders(
        str(GAE_RUN_DIR),
        project_root=str(PROJECT_ROOT),
        prefer_best_validation=True,
        device="cpu",
    )
    wrapper_root = LOG_ROOT / "pretrained_evaluation" / "wrapper_logs" / BENCHMARK
    wrapper_factories["gnn_discrete"] = make_agent_training_wrapper_factory(
        environment=BENCHMARK,
        gae_run_dir=str(GAE_RUN_DIR),
        agent_type="discrete",
        algorithm="ppo",
        project_root=str(PROJECT_ROOT),
        logs_dir=str(wrapper_root / "discrete_gnn"),
    )
    wrapper_factories["iterative"] = make_agent_training_wrapper_factory(
        environment=BENCHMARK,
        gae_run_dir=str(GAE_RUN_DIR),
        agent_type="iterative",
        project_root=str(PROJECT_ROOT),
        logs_dir=str(wrapper_root / "iterative"),
    )
    wrapper_factories["projection"] = make_agent_training_wrapper_factory(
        environment=BENCHMARK,
        gae_run_dir=str(GAE_RUN_DIR),
        agent_type="projection",
        algorithm="ppo",
        project_root=str(PROJECT_ROOT),
        logs_dir=str(wrapper_root / "projection"),
    )
    print("Created wrappers correctly!")
else:
    print("No GAE bundle found. GAE-backed agents will be skipped; ordinary discrete PPO can still be evaluated.")


## 4. Load policies with the correct environment wrappers

In [ ]:

import sys
import importlib

# The canonical module used to instantiate your GNN models
canonical_model = importlib.import_module("LaGCORL.gae.model")

GAE = canonical_model.GAEEncoder
VGAE = canonical_model.VGAEEncoder

print("Canonical GAE:", GAE)

# Find EVERY loaded copy of agent_training_wrapper
patched_modules = []

for module_name, module in list(sys.modules.items()):

    if module is None:
        continue

    if module_name.endswith("agent_training_wrapper"):

        print("Patching module:", module_name)

        # Patch module-level references
        module.GAEEncoder = GAE
        module.VGAEEncoder = VGAE

        # Patch the actual encode function's global namespace
        wrapper_class = getattr(module, "AgentTrainingWrapper", None)

        if wrapper_class is not None:

            encode_function = wrapper_class.encode
            encode_globals = encode_function.__globals__

            encode_globals["GAEEncoder"] = GAE
            encode_globals["VGAEEncoder"] = VGAE

            print(
                "  GAE identity:",
                encode_globals["GAEEncoder"] is GAE
            )

            patched_modules.append(module_name)

# Normalize the duplicate model module reference
sys.modules["gae.model"] = canonical_model

if "gae" in sys.modules:
    sys.modules["gae"].model = canonical_model

print("\nPatched modules:", patched_modules)
print("Number of patched modules:", len(patched_modules))

assert len(patched_modules) > 0
print("\nSUCCESS: all discovered wrapper modules patched.")


In [ ]:
models = {}
load_errors = {}

# GNN+discrete needs the padding config saved by the matching training run.
gnn_test_setup = None
if ARTIFACTS["gnn_discrete"] is not None and "gnn_discrete" in wrapper_factories:
    padding_path = ARTIFACTS["gnn_discrete"].parent / "padding_config.json"
    if padding_path.is_file():
        padding_config = json.loads(padding_path.read_text(encoding="utf-8"))
        gnn_eval_paths = create_run_layout(
            LOG_ROOT, "pretrained_gnn_discrete_evaluation", environment=BENCHMARK
        )
        gnn_test_setup = build_padded_gnn_discrete_corpus(
            project_root=PROJECT_ROOT,
            corpus_dir=TEST_CORPUS_DIR,
            env_ids=TEST_ENV_IDS,
            train_env_id=TEST_ENV_IDS[0],
            wrapper_factory=wrapper_factories["gnn_discrete"],
            log_root=LOG_ROOT,
            run_dir=gnn_eval_paths["run"],
            environment=BENCHMARK,
            padding_config=padding_config,
        )
        try:
            models["gnn_discrete"] = PPO.load(
                str(ARTIFACTS["gnn_discrete"]),
                env=gnn_test_setup["env"],
                device="cpu",
            )
            print("Loaded GNN+discrete PPO.")
        except Exception as exc:
            load_errors["gnn_discrete"] = f"{type(exc).__name__}: {exc}"
            print("Skipping GNN+discrete PPO:", load_errors["gnn_discrete"])
    else:
        print("Skipping GNN+discrete PPO: matching padding_config.json was not found.")

# Iterative iDQN and projection PPO are loaded against their own GAE-backed wrappers.
if ARTIFACTS["iterative"] is not None and "iterative" in wrapper_factories:
    try:
        env = DummyVecEnv([
            lambda: wrapper_factories["iterative"](
                load_benchmark_env(TEST_CORPUS_DIR, TEST_ENV_IDS[0], BENCHMARK)
            )
        ])
        model_class = _get_model_class("idqn")
        models["iterative"] = model_class.load(str(ARTIFACTS["iterative"]), env=env, device="cpu")
        print("Loaded iterative iDQN.")
    except Exception as exc:
        load_errors["iterative"] = f"{type(exc).__name__}: {exc}"
        print("Skipping iterative iDQN:", load_errors["iterative"])

if ARTIFACTS["projection"] is not None and "projection" in wrapper_factories:
    try:
        env = DummyVecEnv([
            lambda: wrapper_factories["projection"](
                load_benchmark_env(TEST_CORPUS_DIR, TEST_ENV_IDS[0], BENCHMARK)
            )
        ])
        models["projection"] = PPO.load(str(ARTIFACTS["projection"]), env=env, device="cpu")
        print("Loaded projection PPO.")
    except Exception as exc:
        load_errors["projection"] = f"{type(exc).__name__}: {exc}"
        print("Skipping projection PPO:", load_errors["projection"])

print("Loaded agents:", list(models))


## 5. Evaluate every compatible policy on the same scenarios

In [ ]:
evaluation_root = (
    LOG_ROOT / "pretrained_evaluation"
    / f"{BENCHMARK}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
)
evaluation_root.mkdir(parents=True, exist_ok=True)
evaluation_results = {}

if "gnn_discrete" in models and gnn_test_setup is not None:
    gnn_frames = [
        evaluate_padded_gnn_policy(
            models["gnn_discrete"],
            env_id=env_id,
            train_env_id=TEST_ENV_IDS[0],
            wrapped_corpus_dir=gnn_test_setup["wrapped_corpus_dir"],
            padding_config=gnn_test_setup["padding_config"],
            log_dir=evaluation_root,
            n_episodes=EVAL_EPISODES,
            seed=EVAL_SEEDS[0],  # same episode seeds (101..110) used by the other agents
            label="GNN + padded discrete PPO",
        )
        for env_id in TEST_ENV_IDS
    ]
    evaluation_results["gnn_discrete"] = {
        "episodes": pd.concat(gnn_frames, ignore_index=True),
    }

for key, label in (("iterative", "iterative iDQN"), ("projection", "projection PPO")):
    if key in models:
        evaluation_results[key] = benchmark_on_corpus(
            models[key],
            corpus_dir=TEST_CORPUS_DIR,
            env_ids=TEST_ENV_IDS,
            test_agent_on_env=test_agent_on_env,
            n_episodes=EVAL_EPISODES,
            seeds=EVAL_SEEDS,
            wrapper_factory=wrapper_factories[key],
            test_metrics_dir=evaluation_root / key,
            label=label,
            verbose=0,
        )

if not evaluation_results:
    raise RuntimeError(
        "No policy could be evaluated. Review the artifact paths and load errors above; "
        "the benchmark environment must match the checkpoint's observation/action schema."
    )

print("Evaluation output:", evaluation_root)
for name, result in evaluation_results.items():
    print(f"\n{name}:")
    if "summary" in result:
        display(pd.DataFrame([result["summary"]]))
    else:
        display(result["episodes"].groupby("agent").mean())


## 6. Combine, normalize, and export the results

The final comparison follows the training notebook: metrics from all agents are normalized to canonical `agent_*` names for comparison only. The combined episode CSV preserves original column names, including unprefixed metrics from GNN-discrete evaluation.


In [ ]:
# delete wrapped corpus_dir if it exists and all its files
if gnn_test_setup is not None and gnn_test_setup.get("wrapped_corpus_dir") is not None:
    wrapped_corpus_dir = gnn_test_setup["wrapped_corpus_dir"]
    if wrapped_corpus_dir.is_dir():
        for file in wrapped_corpus_dir.glob("*"):
            file.unlink()
        wrapped_corpus_dir.rmdir()
        print(f"Deleted temporary wrapped corpus directory: {wrapped_corpus_dir}")

In [ ]:
# Combine episode-level results, preserving each evaluator's original metric names.
episode_frames = []
for name, result in evaluation_results.items():
    frame = result["episodes"].copy()
    if "agent" not in frame.columns:
        frame.insert(0, "agent", name)
    episode_frames.append(frame)

all_episodes = pd.concat(episode_frames, ignore_index=True, sort=False)
combined_csv = evaluation_root / "episode_metrics_all_agents.csv"
all_episodes.to_csv(combined_csv, index=False)

# Match the comparison logic used in the training notebook.
# Iterative/projection/discrete evaluators may use "agent_*" names; the
# GNN-discrete evaluator may expose the equivalent metrics without that prefix.
agent_frames = {
    label: result["episodes"].copy()
    for label, result in evaluation_results.items()
}

if BENCHMARK == "maxcut":
    # Metrics to compare (canonical names).
    expected_metrics = [
        "agent_current_cut",
        "agent_relative_performance",
        "agent_partition0_size",
        "agent_partition1_size",
        "agent_num_steps",
    ]
    
    # Map metric names that may appear without the "agent_" prefix.
    metric_aliases = {
        "current_cut": "agent_current_cut",
        "relative_performance": "agent_relative_performance",
        "partition0_size": "agent_partition0_size",
        "partition1_size": "agent_partition1_size",
        "num_steps": "agent_num_steps",
    }
elif BENCHMARK == "vmp":
    expected_metrics = [
        "agent_weighted_sum_difference",
        "agent_norm_avg_util_difference",
        "agent_norm_avg_power_difference",
        "agent_norm_avg_security_risk_difference",
        "agent_norm_avg_load_difference",
        "agent_norm_packing_efficiency_difference",
        "agent_vm_density_difference",
        "agent_active_pms_difference",
        "agent_actual_valid_movements",
    ]
    
    metric_aliases = {
        "weighted_sum_difference": "agent_weighted_sum_difference",
        "norm_avg_util_difference": "agent_norm_avg_util_difference",
        "norm_avg_power_difference": "agent_norm_avg_power_difference",
        "norm_avg_security_risk_difference": "agent_norm_avg_security_risk_difference",
        "norm_avg_load_difference": "agent_norm_avg_load_difference",
        "norm_packing_efficiency_difference": "agent_norm_packing_efficiency_difference",
        "vm_density_difference": "agent_vm_density_difference",
        "active_pms_difference": "agent_active_pms_difference",
        "actual_valid_movements": "agent_actual_valid_movements",
    }
else:
    raise ValueError(f"Unknown benchmark: {BENCHMARK}")


for label, frame in agent_frames.items():
    rename_map = {
        alias: canonical
        for alias, canonical in metric_aliases.items()
        if alias in frame.columns and canonical not in frame.columns
    }
    agent_frames[label] = frame.rename(columns=rename_map)

missing = {
    label: [metric for metric in expected_metrics if metric not in frame.columns]
    for label, frame in agent_frames.items()
}
for label, absent in missing.items():
    if absent:
        print(f"WARNING — {label} is missing comparison metrics: {absent}")

common_metrics = [
    metric for metric in expected_metrics
    if all(metric in frame.columns for frame in agent_frames.values())
]

if common_metrics:
    comparison_table = pd.DataFrame({
        label: frame[common_metrics].mean(numeric_only=True)
        for label, frame in agent_frames.items()
    }).T
    comparison_table.index.name = "agent"

    display(comparison_table)

    comparison_csv = evaluation_root / "shared_metric_comparison.csv"
    comparison_table.to_csv(comparison_csv)
    print("Saved shared-metric comparison:", comparison_csv)
else:
    comparison_table = pd.DataFrame()
    print(
        "No shared MaxCut comparison metrics are available across all evaluated agents. "
        "The episode-level CSV is still saved. In particular, check whether "
        "evaluate_padded_gnn_policy() returns the environment metrics (not only "
        "episode_reward/episode_length) and whether their names match the aliases above."
    )

# Also save a general numeric summary for quick diagnostics; identifier columns are excluded.
identifier_columns = {
    "agent", "corpus_env_id", "episode", "seed", "env_id", "scenario_id"
}
summary_rows = []
for name, result in evaluation_results.items():
    frame = result["episodes"].copy()
    numeric_columns = [
        col for col in frame.select_dtypes(include=[np.number]).columns
        if col not in identifier_columns
    ]
    row = {"agent": name}
    if numeric_columns:
        row.update(frame[numeric_columns].mean(numeric_only=True).to_dict())
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows).set_index("agent")
summary_csv = evaluation_root / "aggregate_summary.csv"
summary_table.to_csv(summary_csv)

print("Saved episode metrics:", combined_csv)
print("Saved aggregate numeric summary:", summary_csv)